In [7]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

df_raw = pd.read_csv('flight_dataset.csv')
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head(3)

Shape: (10683, 14)
Columns: ['Airline', 'Source', 'Destination', 'Total_Stops', 'Price', 'Date', 'Month', 'Year', 'Dep_hours', 'Dep_min', 'Arrival_hours', 'Arrival_min', 'Duration_hours', 'Duration_min']


,Airline,Source,Destination,Total_Stops,Price,Date,Month,Year,Dep_hours,Dep_min,Arrival_hours,Arrival_min,Duration_hours,Duration_min
0,IndiGo,Banglore,New Delhi,0,3897,24,3,2019,22,20,1,10,2,50
1,Air India,Kolkata,Banglore,2,7662,1,5,2019,5,50,13,15,7,25
2,Jet Airways,Delhi,Cochin,2,13882,9,6,2019,9,25,4,25,19,0


Preprocessing

In [8]:
df = df_raw.copy()

Cleaning functions

Handle missing values.

In [9]:
def handle_missing_values(dataframe, strategy='drop', fill_value=None):
    """
    Detects rows with NaN values and either drops them or fills them.
    Shows affected rows (with row numbers) if issue is found.
    Returns (cleaned_df, issue_found_bool).
    """
    missing_mask = dataframe.isnull().any(axis=1)
    found = missing_mask.any()
    if found:
        print(f"Issue EXISTS — {missing_mask.sum()} rows have missing values.")
        print("Affected rows (before cleaning, row numbers shown as index):")
        print(dataframe[missing_mask].head(10).to_string())
        if strategy == 'drop':
            return dataframe.dropna().reset_index(drop=True), True
        else:
            return dataframe.fillna(fill_value), True
    else:
        print("Issue NOT present in real dataset — demonstrating on made-up example.")
        return dataframe, False

df, found1 = handle_missing_values(df.copy())

#Made up example

print("\n--- Made-up example (8 rows with missing values) ---")
made_up1 = pd.DataFrame({
    'Airline':     ['IndiGo', None, 'Air India', 'SpiceJet', None, 'Vistara', 'GoAir', 'Air Asia'],
    'Source':      ['Delhi', 'Mumbai', 'Banglore', 'Delhi', 'Chennai', None, 'Mumbai', 'Kolkata'],
    'Price':       [3897, 7662, 13882, 4500, None, 8200, 3100, None]
}, index=range(8))

missing_rows = made_up1[made_up1.isnull().any(axis=1)]
print("Rows with missing values (row numbers included):")
print(missing_rows.to_string())
made_up1_clean = made_up1.dropna().reset_index(drop=True)
print("\nAfter dropping rows with missing values:")
print(made_up1_clean.to_string())

Issue NOT present in real dataset — demonstrating on made-up example.

--- Made-up example (8 rows with missing values) ---
Rows with missing values (row numbers included):
    Airline   Source   Price
1      None   Mumbai  7662.0
4      None  Chennai     NaN
5   Vistara     None  8200.0
7  Air Asia  Kolkata     NaN

After dropping rows with missing values:
     Airline    Source    Price
0     IndiGo     Delhi   3897.0
1  Air India  Banglore  13882.0
2   SpiceJet     Delhi   4500.0
3      GoAir    Mumbai   3100.0


Remove Duplicates

In [10]:
def remove_duplicates(dataframe, subset=None):
    """
    Detects and removes duplicate rows (keeps first occurrence).
    Shows the duplicate rows with row numbers before removal.
    Returns (cleaned_df, issue_found_bool).
    """
    dup_mask = dataframe.duplicated(subset=subset, keep='first')
    found = dup_mask.any()
    if found:
        print(f"Issue EXISTS — {dup_mask.sum()} duplicate rows found in dataset.")
        print("Duplicate rows (before removal, row numbers shown as index) — first 10:")
        print(dataframe[dup_mask].head(10).to_string())
        cleaned = dataframe[~dup_mask].reset_index(drop=True)
        print(f"\nShape before: {dataframe.shape}  →  Shape after: {cleaned.shape}")
        return cleaned, True
    else:
        print("Issue NOT present in real dataset — demonstrating on made-up example.")
        return dataframe, False

df, found2 = remove_duplicates(df.copy())

Issue EXISTS — 222 duplicate rows found in dataset.
Duplicate rows (before removal, row numbers shown as index) — first 10:
          Airline    Source Destination  Total_Stops  Price  Date  Month  Year  Dep_hours  Dep_min  Arrival_hours  Arrival_min  Duration_hours  Duration_min
683   Jet Airways     Delhi      Cochin            2  13376     1      6  2019         14       35              4           25              13            50
1061    Air India     Delhi      Cochin            2  10231    21      5  2019         22        0             19           15              21            15
1348    Air India     Delhi      Cochin            2  12392    18      5  2019         17       15             19           15              26             0
1418  Jet Airways     Delhi      Cochin            2  10368     6      6  2019          5       30              4           25              22            55
1674       IndiGo  Banglore   New Delhi            0   7303    24      3  2019         18  

Handle outliers.

In [11]:
def handle_outliers(dataframe, column, factor=1.5):
    """
    Detects outliers using the IQR method and caps them at the lower/upper fence.
    Shows affected rows with row numbers before capping.
    Returns (cleaned_df, issue_found_bool).
    """
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR

    outlier_mask = (dataframe[column] < lower) | (dataframe[column] > upper)
    found = outlier_mask.any()
    if found:
        print(f"Issue EXISTS — {outlier_mask.sum()} outliers found in '{column}'.")
        print(f"  IQR fences: lower = {lower:.2f},  upper = {upper:.2f}")
        print(f"  Range before capping: [{dataframe[column].min():.2f}, {dataframe[column].max():.2f}]")
        print("\nOutlier rows (before capping, row numbers shown as index) — first 10:")
        print(dataframe[outlier_mask][[column]].head(10).to_string())
        dataframe.loc[dataframe[column] < lower, column] = lower
        dataframe.loc[dataframe[column] > upper, column] = upper
        print(f"\nRange after capping: [{dataframe[column].min():.2f}, {dataframe[column].max():.2f}]")
        return dataframe, True
    else:
        print(f"No outliers found in '{column}'.")
        return dataframe, False

df, found3 = handle_outliers(df.copy(), column='Price')

Issue EXISTS — 94 outliers found in 'Price'.
  IQR fences: lower = -5459.00,  upper = 23029.00
  Range before capping: [1759.00, 79512.00]

Outlier rows (before capping, row numbers shown as index) — first 10:
     Price
123  27430
396  36983
486  26890
510  26890
597  25139
628  27210
657  52229
784  26743
825  26890
935  25735

Range after capping: [1759.00, 23029.00]


Normalize mixed-format dates to a single standard : YYYY-MM-DD.

In [12]:
def normalize_dates(dataframe, day_col, month_col, year_col, new_col='journey_date'):
    """
    Combines separate day/month/year integer columns into a single YYYY-MM-DD string column.
    Also demonstrates handling mixed string formats when given string input.
    Shows a sample of the new column after creation.
    """
    dataframe[new_col] = pd.to_datetime(
        dict(year=dataframe[year_col],
             month=dataframe[month_col],
             day=dataframe[day_col])
    ).dt.strftime('%Y-%m-%d')
    return dataframe

# Apply to real dataset
df = normalize_dates(df.copy(), day_col='Date', month_col='Month', year_col='Year')
print("Sample of new 'journey_date' column (5 rows):")
print(df[['Date','Month','Year','journey_date']].head(5).to_string())

# Made-up example
print("\n--- Made-up example: mixed date string formats ---")
made_up4 = pd.DataFrame({
    'FlightID':    ['F1','F2','F3','F4','F5','F6'],
    'RawDate':     ['24/3/2019','2019-05-01','June 9, 2019',
                    '2019/04/15','10-May-2019','06.20.2019'],
    'Price':       [3897, 7662, 13882, 4500, 8200, 6100]
})
print("Before normalization:")
print(made_up4[['FlightID','RawDate']].to_string())
made_up4['RawDate'] = pd.to_datetime(made_up4['RawDate'], format='mixed').dt.strftime('%Y-%m-%d')
print("\nAfter normalization — sample of affected column (all 6 rows):")
print(made_up4[['FlightID','RawDate']].to_string())

Sample of new 'journey_date' column (5 rows):
   Date  Month  Year journey_date
0    24      3  2019   2019-03-24
1     1      5  2019   2019-05-01
2     9      6  2019   2019-06-09
3    12      5  2019   2019-05-12
4     1      3  2019   2019-03-01

--- Made-up example: mixed date string formats ---
Before normalization:
  FlightID       RawDate
0       F1     24/3/2019
1       F2    2019-05-01
2       F3  June 9, 2019
3       F4    2019/04/15
4       F5   10-May-2019
5       F6    06.20.2019

After normalization — sample of affected column (all 6 rows):
  FlightID     RawDate
0       F1  2019-03-24
1       F2  2019-05-01
2       F3  2019-06-09
3       F4  2019-04-15
4       F5  2019-05-10
5       F6  2019-06-20


Standardize categorical values (e.g., "M", "Male", "male" → "Male").

In [13]:
def standardize_categorical(dataframe, columns, case='title'):
    """
    Strips leading/trailing whitespace and applies uniform casing to
    categorical text columns. Shows a sample of affected columns after cleaning.
    Returns (cleaned_df, set_of_changed_indices).
    """
    changed_rows = set()
    for col in columns:
        original = dataframe[col].astype(str).copy()
        if case == 'title':
            cleaned = dataframe[col].astype(str).str.strip().str.title()
        elif case == 'lower':
            cleaned = dataframe[col].astype(str).str.strip().str.lower()
        else:
            cleaned = dataframe[col].astype(str).str.strip().str.upper()
        diff = cleaned != original
        changed_rows.update(dataframe[diff].index.tolist())
        dataframe[col] = cleaned
    return dataframe, changed_rows

# Checking dataset
cat_cols = ['Airline', 'Source', 'Destination']
any_issue = False
for col in cat_cols:
    issue = (df[col].astype(str) != df[col].astype(str).str.strip().str.title()).any()
    if issue:
        print(f"Case issue found in column: {col}")
        any_issue = True
if not any_issue:
    print("No case/whitespace issues in real dataset — demonstrating on made-up example.")

# Made-up example
print("\n--- Made-up example (mixed case and whitespace) ---")
made_up5 = pd.DataFrame({
    'Airline':     ['  indigo', 'AIR INDIA', 'Jet airways ', 'SPICEJET', 'goair', 'vistara'],
    'Source':      ['delhi', 'MUMBAI', 'Banglore ', 'chennai', 'KOLKATA', 'delhi'],
    'Destination': ['COCHIN', 'delhi', 'New delhi', 'BANGLORE', 'mumbai', 'HYDERABAD']
})
print("Before:")
print(made_up5.to_string())
made_up5_clean, changed = standardize_categorical(made_up5.copy(), ['Airline','Source','Destination'])
print(f"\nAfter standardization — sample of affected columns ({len(changed)} rows changed):")
print(made_up5_clean.to_string())

# Apply to real dataset to keep values consistent
df, _ = standardize_categorical(df, cat_cols)
print("\nTitle-case + strip applied to real dataset. Sample:")
print(df[cat_cols].head(5).to_string())

Case issue found in column: Airline

--- Made-up example (mixed case and whitespace) ---
Before:
        Airline     Source Destination
0        indigo      delhi      COCHIN
1     AIR INDIA     MUMBAI       delhi
2  Jet airways   Banglore    New delhi
3      SPICEJET    chennai    BANGLORE
4         goair    KOLKATA      mumbai
5       vistara      delhi   HYDERABAD

After standardization — sample of affected columns (6 rows changed):
       Airline    Source Destination
0       Indigo     Delhi      Cochin
1    Air India    Mumbai       Delhi
2  Jet Airways  Banglore   New Delhi
3     Spicejet   Chennai    Banglore
4        Goair   Kolkata      Mumbai
5      Vistara     Delhi   Hyderabad

Title-case + strip applied to real dataset. Sample:
       Airline    Source Destination
0       Indigo  Banglore   New Delhi
1    Air India   Kolkata    Banglore
2  Jet Airways     Delhi      Cochin
3       Indigo   Kolkata    Banglore
4       Indigo  Banglore   New Delhi


In [14]:
import os
df.to_csv('diy_dataset.csv', index=False)
size_kb = os.path.getsize('diy_dataset.csv') / 1024
print(f"Cleaned dataset saved → diy_dataset.csv")
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  File size: {size_kb:.1f} KB")

Cleaned dataset saved → diy_dataset.csv
  Shape: 10,461 rows × 15 columns
  File size: 716.4 KB


Markdown explanation

Which of the five issues were present in your dataset? (List and summarize the fixes you applied.)

Duplicates and outliers were found. To be exact, 222 duplicate rows were found and removed while 94 rows had drastic price values which served as outliers. The outliers' values were changed to fit either the lower or upper INR value within the IQR range.

Which issues were not present, and what made-up example did you construct to demonstrate your pipeline function for them?

There were no missing values, mixed-format dates, and case standardization that had to be done. For the missing values, 8 rows were made up with a missing Airline, Source, and Price values to demonstrate the detection. The dataset stores date parts as separate integers rather than mixed-format strings. So, they were combined to form a journey_date column. 6 rows were made up with 6 different raw string date formats. All of the columns were consistently formatted so there was no need for case standardization. 6 rows were made up as the example to showcase normalization of the rows which had mixed casing and whitespace padding. .str.strip().str.title() normalization was demonstrated through this 6-row example.

Analysis

In [ ]:
df = pd.read_csv('diy_dataset.csv')
df['journey_date'] = pd.to_datetime(df['journey_date'])
print("Shape:", df.shape)
df.head(3)

Top-N within each group

In [ ]:
avg_price = (df.groupby(['Source', 'Airline'])['Price']
               .mean()
               .reset_index()
               .rename(columns={'Price': 'Avg Price (INR)'}))

top3 = (avg_price
        .sort_values('Avg Price (INR)', ascending=False)
        .groupby('Source')
        .head(3)
        .sort_values(['Source', 'Avg Price (INR)'], ascending=[True, False]))

top3['Avg Price (INR)'] = top3['Avg Price (INR)'].round(2)
print("Top 3 Airlines by Average Price for each Source City:")
print(top3.to_string(index=False))

Bin comparison

In [ ]:
bins   = [0, 5, 12, df['Duration_hours'].max()]
labels = ['Short (1–5 hrs)', 'Medium (6–12 hrs)', 'Long (13+ hrs)']
df['Duration_Bin'] = pd.cut(df['Duration_hours'], bins=bins, labels=labels)

bin_price = (df.groupby('Duration_Bin', observed=True)['Price']
               .agg(['mean', 'count'])
               .rename(columns={'mean': 'Avg Price (INR)', 'count': 'Count'}))

print("Average Price by Flight Duration Bin:")
print(bin_price.round(2).to_string())

Conditional Aggregation

In [ ]:
nonstop = df[df['Total_Stops'] == 0]
avg_nonstop = (nonstop.groupby('Airline')['Price']
                      .mean()
                      .rename('Avg Non-Stop Price (INR)')
                      .sort_values(ascending=False)
                      .round(2))

print(f"Non-stop flights: {len(nonstop):,} rows")
print("\nAverage Non-Stop Price by Airline:")
print(avg_nonstop.to_string())

Change over Time

In [ ]:
df['YearMonth'] = df['journey_date'].dt.to_period('M')
monthly_avg = (df.groupby('YearMonth')['Price']
                 .mean()
                 .rename('Avg Price (INR)')
                 .round(2))

print("Average Flight Price by Month:")
print(monthly_avg.to_string())